In [1]:
import openai

print(openai.__version__)

2.14.0


In [2]:
import os
import sys

os.chdir("..")
sys.path.append("src/")

In [3]:
from openai import OpenAI

model = "gpt-5.1"
client = OpenAI()

user_prompt = "Hi, my name is Maliheh"
response = client.responses.create(model=model, input=user_prompt)

print(response.output_text)

Hi Maliheh, nice to meet you.  
How can I help you today?


In [4]:
from pydantic import BaseModel


class EventExtractInfo(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.responses.parse(
    model=model,
    input=[
        {"role": "system", "content": "Extract the event information"},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=EventExtractInfo,
)

print(response.output_parsed)

name='science fair' date='Friday' participants=['Alice', 'Bob']


In [5]:
model = "gpt-4.1-mini"

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": "https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg",
                },
            ],
        }
    ],
)

print(response.output_text)

The image shows a variety of foods arranged on a light blue surface. The items include:

- Broccoli and broccolini
- Red cabbage
- Brown rice in a small bowl
- Avocado (one whole and one half)
- Pistachio nuts on a small plate
- Dark chocolate bars
- Bok choy
- Three eggs
- Raw salmon fillet on a white plate
- Sauerkraut in a small bowl
- Kimchi in a small bowl
- Two red apples
- Chia seeds in a small bowl
- A bowl of grains (possibly barley or wheat berries) 

These ingredients appear to be healthy, nutrient-rich foods.


In [6]:
import base64

model = "gpt-4.1"


def encode_image_by_path(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

print(response.output_text)

This image features a variety of healthy foods commonly associated with a balanced diet, gut health, or nutritious eating. Here is a list of what is visible in the image:

- Red cabbage (top left)
- Broccoli or broccolini (left center)
- Brown rice (top in a bowl)
- Bok choy (center, leafy greens)
- Salmon fillet (top right, on a plate)
- Whole grain (bottom right in a bowl, could be farro or wheat berries)
- Eggs (center right)
- Sauerkraut (in a bowl next to eggs)
- Avocado (whole and halved)
- Chia seeds (in a small bowl)
- Pistachios (shelled, on a plate)
- Dark chocolate bars (center bottom)
- Kimchi (fermented vegetables, in a bowl)
- Apples (bottom right, two whole apples)

These foods provide a wide range of nutrients and are often included in diets for their positive effects on health, including gut health due to fermented foods like sauerkraut and kimchi.


In [7]:
# lets you convert files (like images) into Base64 text


def encode_image_by_path(image_path):
    # Step 1: Open the image as bytes (binary data)
    # rb: read binary (images are not text, they're raw bytes)
    image_file = open(image_path, "rb")

    # Step 2: Read all the bytes from the image
    image_bytes = image_file.read()

    # Step 3: Convert those bytes into Base64 bytes
    base64_bytes = base64.b64encode(image_bytes)

    # Step 4: Convert Base64 bytes into a normal string
    base64_string = base64_bytes.decode("utf-8")

    # Step 5: Return the final Base64 string
    return base64_string

In [8]:

raw_data = b"\x89PNG\r\n\x1a\n"  # This is how real image bytes often look
print("RAW BYTES:", raw_data)

# Step 1: Base64 encode the raw bytes
base64_bytes = base64.b64encode(raw_data)
print("BASE64 (bytes):", base64_bytes)

# Step 2: Convert Base64 bytes → string
base64_string = base64_bytes.decode("utf-8")
print("BASE64 (string):", base64_string)

RAW BYTES: b'\x89PNG\r\n\x1a\n'
BASE64 (bytes): b'iVBORw0KGgo='
BASE64 (string): iVBORw0KGgo=


In [9]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_path
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

name='Healthy Whole Food Spread' ingredients=[Ingredient(ingredient_name='Red cabbage', portiont='1/4 head'), Ingredient(ingredient_name='Broccolini', portiont='4 stalks'), Ingredient(ingredient_name='Bok choy', portiont='3 small heads'), Ingredient(ingredient_name='Salmon fillet', portiont='1 piece, about 4-6 oz'), Ingredient(ingredient_name='Brown rice', portiont='1/2 cup, uncooked'), Ingredient(ingredient_name='Farro', portiont='1/2 cup, uncooked'), Ingredient(ingredient_name='Avocado', portiont='1 whole'), Ingredient(ingredient_name='Eggs', portiont='3 whole eggs'), Ingredient(ingredient_name='Chia seeds', portiont='2 tsp'), Ingredient(ingredient_name='Pistachios', portiont='1/4 cup'), Ingredient(ingredient_name='Dark chocolate', portiont='3 oz (about 5 squares)'), Ingredient(ingredient_name='Kimchi', portiont='1/2 cup'), Ingredient(ingredient_name='Sauerkraut', portiont='1/2 cup'), Ingredient(ingredient_name='Apples', portiont='2 whole')]


In [10]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_image_url(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    image_url=image_url,
    response_format=IngredientsResponse,
)

print(response)

name='Vietnamese Hu Tieu Nam Vang (Seafood Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice Noodles', portiont='1 bowl'), Ingredient(ingredient_name='Shrimp', portiont='2 pieces'), Ingredient(ingredient_name='Pork slices', portiont='4-5 pieces'), Ingredient(ingredient_name='Quail Eggs', portiont='3 pieces'), Ingredient(ingredient_name='Fresh herbs (such as cilantro, mint)', portiont='1 small handful'), Ingredient(ingredient_name='Chopped green onions', portiont='1 tablespoon'), Ingredient(ingredient_name='Sliced red chili', portiont='1 teaspoon'), Ingredient(ingredient_name='Savory soup broth', portiont='1.5 cups')]


In [11]:
from pydantic import BaseModel

from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_url
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
base64_image = encode_image_by_url(image_url)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

name='Vietnamese Noodle Soup (Bánh Canh Tôm Thịt)' ingredients=[Ingredient(ingredient_name='Wide rice noodles (bánh canh)', portiont='1 serving'), Ingredient(ingredient_name='Shrimp', portiont='2 large pieces'), Ingredient(ingredient_name='Pork slices', portiont='3-4 slices'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Fresh herbs (mint, cilantro, scallion)', portiont='small handful'), Ingredient(ingredient_name='Sliced red chili', portiont='1 tablespoon'), Ingredient(ingredient_name='Broth (pork/shrimp based)', portiont='1 cup'), Ingredient(ingredient_name='Fried garlic/onion', portiont='1 tablespoon')]


In [12]:
from services.analysis.ingredients import IngredientsAnalyzer

ingredients_analyzer = IngredientsAnalyzer()

image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
result = ingredients_analyzer.analyze(image_url)
print(result)

name='Vietnamese Hu Tieu Nam Vang (Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 serving (bottom layer)'), Ingredient(ingredient_name='Prawns/Shrimp', portiont='2 large pieces'), Ingredient(ingredient_name='Quail eggs', portiont='4 pieces'), Ingredient(ingredient_name='Pork slices', portiont='Several slices (about 3-4 pieces)'), Ingredient(ingredient_name='Fresh herbs (cilantro, mint)', portiont='A small handful'), Ingredient(ingredient_name='Chopped green onions', portiont='1 tablespoon, sprinkled on top'), Ingredient(ingredient_name='Sliced red chili', portiont='1 tablespoon, thinly sliced'), Ingredient(ingredient_name='Soup broth', portiont='Enough to cover noodles (about 1-2 cups)')]


In [13]:
from services.analysis.nutrients import NutrientsAnalyzer
from services.analysis.schemas import Ingredient

nutrients_analyzer = NutrientsAnalyzer()

ingredients = [
    Ingredient(ingredient_name="Rice noodles", portiont="1 serving"),
    Ingredient(ingredient_name="Shrimp", portiont="2 pieces"),
    Ingredient(ingredient_name="Quail eggs", portiont="4 pieces"),
    Ingredient(ingredient_name="Pork slices", portiont="4-5 slices"),
    Ingredient(ingredient_name="Fresh herbs (such as mint)", portiont="A handful"),
    Ingredient(ingredient_name="Green onions", portiont="1 tablespoon, chopped"),
    Ingredient(ingredient_name="Red chili pepper", portiont="1 tablespoon, sliced"),
    Ingredient(ingredient_name="Broth (pork or seafood-based)", portiont="1 cup"),
]
result = nutrients_analyzer.analyze(ingredients)
print(result)

total_calories=410 total_protein_g=28.0 total_carbohydrates_g=56.0 total_fats_g=10.0 total_fiber_g=3.0
